## knn-v4.ipynb

Extends V3 by adding a dedicated release year feature block produced by
`weights-year.ipynb`.

| New block | Columns | Signal |
|---|---|---|
| `album_year_matrix.npz` | 1 | Min-max scaled release year (imputed) |

All V3 weights are carried forward. `W_YEAR` is a new tuneable weight;
start at 1.0 and tune via `tune-weights.ipynb` or `tune-base-weights.ipynb`.

### Note on track stats overlap
The existing `album_track_stats_matrix.npz` includes `first_release_year`
as one of its 12 columns (effective contribution ≈ 0.25/12 at current weight).
The dedicated year block provides independent weight control and uses the
higher-coverage imputed column. Rebuilding track stats without year is
optional — the overlap is small given track_stats weight of 0.25.

In [1]:
import os
import json
import pickle
import numpy as np
import pandas as pd
import joblib
from scipy.sparse import csr_matrix, hstack, load_npz, save_npz
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import normalize

DATA_DIR     = '../data'
FEATURES_DIR = f'{DATA_DIR}/features'
os.makedirs(f'{DATA_DIR}/model_v4', exist_ok=True)

# Load best V3 weights
with open(f'{DATA_DIR}/best_weights.json') as f:
    best_weights = json.load(f)

V3 = best_weights['v3']
print('V3 base weights loaded:')
for k, v in V3.items():
    print(f'  {k:<18} = {v}')

V3 base weights loaded:
  W_TAGS             = 1.0
  W_LABELS           = 0.32472055144502865
  W_TYPES            = 3.5067764992972172
  W_RATINGS          = 0.1
  W_COUNTRY          = 1.0
  W_TRACK_STATS      = 0.25
  W_ROLE_FAMILY      = 0.05
  W_INSTRUMENT       = 0.03
  W_CONTRIB_CNT      = 0.0


In [2]:
# Load row index and all feature blocks
with open(f'{FEATURES_DIR}/album_ids.pkl', 'rb') as f:
    album_id_order = pickle.load(f)

X_tags        = load_npz(f'{FEATURES_DIR}/album_tags_matrix.npz')
X_labels      = load_npz(f'{FEATURES_DIR}/album_labels_matrix.npz')
X_types       = load_npz(f'{FEATURES_DIR}/album_types_matrix.npz')
X_ratings     = load_npz(f'{FEATURES_DIR}/album_ratings_matrix.npz')
X_country     = load_npz(f'{FEATURES_DIR}/album_country_matrix.npz')
X_track_stats = load_npz(f'{FEATURES_DIR}/album_track_stats_matrix.npz')
X_role_family = load_npz(f'{FEATURES_DIR}/album_role_family_matrix.npz')
X_instrument  = load_npz(f'{FEATURES_DIR}/album_instrument_matrix.npz')
X_contrib_cnt = load_npz(f'{FEATURES_DIR}/album_contributor_counts_matrix.npz')
X_year        = load_npz(f'{FEATURES_DIR}/album_year_matrix.npz')   # new

print('Feature blocks loaded:')
for name, X in [
    ('X_tags', X_tags), ('X_labels', X_labels), ('X_types', X_types),
    ('X_ratings', X_ratings), ('X_country', X_country),
    ('X_track_stats', X_track_stats), ('X_role_family', X_role_family),
    ('X_instrument', X_instrument), ('X_contrib_cnt', X_contrib_cnt),
    ('X_year', X_year),
]:
    print(f'  {name:<16} {str(X.shape):<25} nnz={X.nnz:,}')

Feature blocks loaded:
  X_tags           (1008102, 3041)           nnz=3,004,997
  X_labels         (1008102, 3469)           nnz=402,047
  X_types          (1008102, 10)             nnz=402,047
  X_ratings        (1008102, 1)              nnz=44,334
  X_country        (1008102, 2263)           nnz=883,503
  X_track_stats    (1008102, 12)             nnz=11,139,577
  X_role_family    (1008102, 7)              nnz=1,155,920
  X_instrument     (1008102, 591)            nnz=1,459,235
  X_contrib_cnt    (1008102, 7)              nnz=1,155,920
  X_year           (1008102, 1)              nnz=997,497


In [3]:
# Expand all matrices to full album universe
full_album_ids = pd.Index(
    pd.read_parquet(f'{DATA_DIR}/mb_album.parquet', columns=['id'])['id'].sort_values()
)

all_blocks = {
    'tags': X_tags, 'labels': X_labels, 'types': X_types,
    'ratings': X_ratings, 'country': X_country, 'track_stats': X_track_stats,
    'role_family': X_role_family, 'instrument': X_instrument,
    'contrib_cnt': X_contrib_cnt, 'year': X_year,
}

if len(album_id_order) < len(full_album_ids):
    print(f'Expanding {len(album_id_order):,} → {len(full_album_ids):,} albums...')
    pos    = full_album_ids.get_indexer(album_id_order)
    n_full = len(full_album_ids)
    def _expand(X, row_pos, n):
        coo = X.tocoo()
        return csr_matrix((coo.data, (row_pos[coo.row], coo.col)), shape=(n, X.shape[1]))
    all_blocks = {k: _expand(v, pos, n_full) for k, v in all_blocks.items()}
    album_id_order = full_album_ids.tolist()

X_tags, X_labels, X_types, X_ratings, X_country, X_track_stats, \
    X_role_family, X_instrument, X_contrib_cnt, X_year = all_blocks.values()
print('Expansion done.')

Expanding 1,008,102 → 2,241,402 albums...
Expansion done.


In [4]:
from scipy.sparse import hstack  # re-import in case cell is run standalone

# ── Block weights ─────────────────────────────────────────────────────────
# All V3 weights carried forward from best_weights.json.
# W_YEAR: start at 1.0 — tune via tune-weights.ipynb after building the model.
# A higher value rewards era similarity more strongly;
# try 0.5–3.0 depending on how much year should influence recommendations.
W_TAGS        = V3['W_TAGS']
W_LABELS      = V3['W_LABELS']
W_TYPES       = V3['W_TYPES']
W_RATINGS     = V3['W_RATINGS']
W_COUNTRY     = V3['W_COUNTRY']
W_TRACK_STATS = V3['W_TRACK_STATS']
W_ROLE_FAMILY = V3['W_ROLE_FAMILY']
W_INSTRUMENT  = V3['W_INSTRUMENT']
W_CONTRIB_CNT = V3.get('W_CONTRIB_CNT', 0.0)
W_YEAR        = 1.0   # <-- tune this

X_final_v4 = hstack([
    X_tags        * W_TAGS,
    X_labels      * W_LABELS,
    X_types       * W_TYPES,
    X_ratings     * W_RATINGS,
    X_country     * W_COUNTRY,
    X_track_stats * W_TRACK_STATS,
    X_role_family * W_ROLE_FAMILY,
    X_instrument  * W_INSTRUMENT,
    X_contrib_cnt * W_CONTRIB_CNT,
    X_year        * W_YEAR,
]).tocsr()

print(f'X_final_v4: {X_final_v4.shape[0]:,} albums × {X_final_v4.shape[1]:,} features  '
      f'(nnz={X_final_v4.nnz:,})')
print(f'\nBlock summary:')
for name, X, w in [
    ('tags', X_tags, W_TAGS), ('labels', X_labels, W_LABELS),
    ('types', X_types, W_TYPES), ('ratings', X_ratings, W_RATINGS),
    ('country', X_country, W_COUNTRY), ('track_stats', X_track_stats, W_TRACK_STATS),
    ('role_family', X_role_family, W_ROLE_FAMILY),
    ('instrument', X_instrument, W_INSTRUMENT),
    ('contrib_cnt', X_contrib_cnt, W_CONTRIB_CNT),
    ('year', X_year, W_YEAR),
]:
    print(f'  {name:<16} w={w:<8}  cols={X.shape[1]:,}')

X_final_v4: 2,241,402 albums × 9,402 features  (nnz=20,645,077)

Block summary:
  tags             w=1.0       cols=3,041
  labels           w=0.32472055144502865  cols=3,469
  types            w=3.5067764992972172  cols=10
  ratings          w=0.1       cols=1
  country          w=1.0       cols=2,263
  track_stats      w=0.25      cols=12
  role_family      w=0.05      cols=7
  instrument       w=0.03      cols=591
  contrib_cnt      w=0.0       cols=7
  year             w=1.0       cols=1


In [5]:
# Safe column pruning
col_nnz      = np.diff(X_final_v4.tocsc().indptr)
row_lengths  = np.diff(X_final_v4.indptr)
has_features = row_lengths > 0
nonempty     = np.where(has_features)[0]
col_vals     = col_nnz[X_final_v4.indices]
max_col      = np.zeros(X_final_v4.shape[0], dtype=col_nnz.dtype)
max_col[nonempty] = np.maximum.reduceat(col_vals, X_final_v4.indptr[nonempty])
safe_thresh  = int(max_col[has_features].min())
X_knn_v4     = X_final_v4[:, col_nnz >= safe_thresh]

print(f'Safe threshold  : {safe_thresh}')
print(f'Columns before  : {X_final_v4.shape[1]:,}')
print(f'Columns after   : {X_knn_v4.shape[1]:,}')
print(f'Albums with features: {has_features.sum():,}  ({has_features.mean()*100:.1f}%)')

Safe threshold  : 10
Columns before  : 9,402
Columns after   : 6,460
Albums with features: 1,008,102  (45.0%)


In [6]:
# Subset to annotated albums, L2-normalise, fit
X_ann   = X_knn_v4[has_features].copy()
ids_ann = np.array(album_id_order)[has_features]
np.nan_to_num(X_ann.data, nan=0.0, copy=False)
X_ann.eliminate_zeros()
X_norm_v4 = normalize(X_ann, norm='l2')

print(f'Fitting on {X_norm_v4.shape[0]:,} albums × {X_norm_v4.shape[1]:,} features...')
model_v4 = NearestNeighbors(metric='cosine', algorithm='brute', n_jobs=-1)
model_v4.fit(X_norm_v4)
print('Model v4 fitted.')

Fitting on 1,008,102 albums × 6,460 features...
Model v4 fitted.


In [7]:
# Sanity check + year coverage report
distances, indices = model_v4.kneighbors(X_norm_v4[0], n_neighbors=11)
id2row_v4  = {int(aid): i for i, aid in enumerate(ids_ann)}

print(f'Query album id: {int(ids_ann[0])}')
print(f"\n{'rank':<6} {'album_id':<14} {'distance':>10}")
print('-' * 34)
for rank, (idx, dist) in enumerate(zip(indices[0], distances[0])):
    label = '(query)' if rank == 0 else ''
    print(f"{rank:<6} {int(ids_ann[idx]):<14} {dist:>10.4f}  {label}")

print(f'\nYear block coverage:')
year_nnz = int(X_year[has_features].nnz)
print(f'  Albums with year in annotated set: '
      f'{year_nnz:,} / {has_features.sum():,}  '
      f'({year_nnz/has_features.sum()*100:.1f}%)')

Query album id: 4

rank   album_id         distance
----------------------------------
0      4                  0.0000  (query)
1      576                0.0009  
2      675861             0.0015  
3      243939             0.0019  
4      131486             0.0020  
5      2210947            0.0021  
6      587551             0.0024  
7      100757             0.0031  
8      59014              0.0034  
9      362486             0.0035  
10     36458              0.0036  

Year block coverage:
  Albums with year in annotated set: 997,497 / 1,008,102  (98.9%)


In [8]:
joblib.dump(model_v4,  f'{DATA_DIR}/model_v4/knn_model_v4.joblib')
save_npz(f'{DATA_DIR}/model_v4/X_knn_norm_v4.npz', X_norm_v4)
np.save(f'{DATA_DIR}/model_v4/album_ids_annotated_v4.npy', ids_ann)
np.save(f'{DATA_DIR}/model_v4/has_features_v4.npy', has_features)

# Save W_YEAR alongside best_weights for reference
best_weights['v4'] = {**V3, 'W_YEAR': W_YEAR}
with open(f'{DATA_DIR}/best_weights.json', 'w') as f:
    json.dump(best_weights, f, indent=2)

print('Saved to ../data/model_v4/')
print(f'W_YEAR={W_YEAR} — tune this in tune-weights.ipynb before final deployment')

Saved to ../data/model_v4/
W_YEAR=1.0 — tune this in tune-weights.ipynb before final deployment
